# Исследование детекции и очистки посторонней речи в телефонном звонке

Ноутбук содержит полный воспроизводимый цикл исследования:
1. Загрузка реальных дикторов (`src/data_loader.py`)
2. Физическая симуляция диалога и искажений телефонного тракта (`src/synthesizer.py`)
3. Сравнение предсказаний базового правила (+6 дБ) и нашей модели (`src/detector.py`)
4. Очистка от наводки/эха (`src/cleaner.py`) и прослушивание аудио с замером сходства эмбеддингов
5. Профилирование скорости RTF на 1 ядре CPU (`torch.set_num_threads(1)`)


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))
sys.path.append(str(Path('.').resolve()))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from src.config import SAMPLE_RATE, FRAME_DURATION_SEC, COLLAR_SEC
from src.data_loader import RealSpeechLoader
from src.synthesizer import TelephoneDialogueSynthesizer
from src.baseline import EnergyBaselineDetector
from src.detector import ExtraneousSpeechDetector
from src.cleaner import TelephoneAudioCleaner, compute_spectral_embedding
from src.metrics import evaluate_channels, cosine_similarity, measure_rtf

print('Environment initialized. Sample rate:', SAMPLE_RATE, 'Hz')


Environment initialized. Sample rate: 8000 Hz


## 1. Загрузка реальных речевых данных и синтез телефонного диалога

Берем реальные записи дикторов (`aew` — владелец канала A, `slt` — собеседник B, `bdl` — третий говорящий) и синтезируем сценарий `double_talk`.

In [2]:
loader = RealSpeechLoader(data_root='data/cmuarctic')
pool = loader.build_utterance_pool('aew', 'slt', 'bdl', max_per_speaker=30)
synth = TelephoneDialogueSynthesizer(seed=2026)

res = synth.synthesize_dialogue(utterance_pool=pool, duration_sec=12.0, scenario='double_talk', third_speaker_prob=1.0)
audio = res.audio
sr = res.sample_rate
gt = res.ground_truth_intervals

print(f'Audio shape: {audio.shape}, Duration: {res.duration_sec}s')
print('Metadata:', res.metadata)


KeyError: "Speaker 'aew' not found. Available: []"

## 2. Спектрограммы, осциллограммы и Ground-Truth разметка

In [ ]:
t = np.arange(audio.shape[1]) / sr
fig, axes = plt.subplots(3, 2, figsize=(15, 8), sharex=True, gridspec_kw={'height_ratios': [1.2, 1.2, 1.0]})

# Waveforms
axes[0, 0].plot(t, audio[0], color='#1f77b4', lw=0.8)
axes[0, 0].set_title('Track A (Channel A) - Waveform')
axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(t, audio[1], color='#ff7f0e', lw=0.8)
axes[0, 1].set_title('Track B (Channel B) - Waveform')
axes[0, 1].grid(True, alpha=0.3)

# Spectrograms (300 - 3400 Hz telephone band)
axes[1, 0].specgram(audio[0], Fs=sr, NFFT=256, noverlap=128, cmap='viridis')
axes[1, 0].set_title('Track A - Telephone Spectrogram')
axes[1, 0].set_ylabel('Frequency (Hz)')

axes[1, 1].specgram(audio[1], Fs=sr, NFFT=256, noverlap=128, cmap='viridis')
axes[1, 1].set_title('Track B - Telephone Spectrogram')

# Activity Timelines
color_map = {'own': '#2ca02c', 'leak': '#d62728', 'third': '#9467bd'}
y_positions = {'own': 2, 'leak': 1, 'third': 0}

for i, ch in enumerate(['A', 'B']):
    ax = axes[2, i]
    ax.set_ylim(-0.5, 2.5)
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(['third', 'leak', 'own'])
    ax.set_xlabel('Time (seconds)')
    ax.grid(True, alpha=0.3)
    for seg in gt.get(ch, []):
        lbl = seg['label']
        ax.barh(y_positions.get(lbl, 0), seg['end'] - seg['start'], left=seg['start'], height=0.6,
                color=color_map.get(lbl, 'gray'), alpha=0.85, edgecolor='black')
    ax.set_title(f'Ground-Truth Activity: Track {ch}')

plt.tight_layout()
plt.show()


## 3. Сравнение: Базовое правило (+6 дБ) vs Обученный детектор

Сравниваем качество детекции на сценарии `double_talk` с допуском 50 мс на границах.

In [ ]:
base_detector = EnergyBaselineDetector()
base_preds = base_detector.predict_intervals(audio, sample_rate=sr)

detector = ExtraneousSpeechDetector()
det_preds = detector.predict_intervals(audio, sample_rate=sr)

m_base = evaluate_channels(gt, base_preds, num_frames=res.num_frames, collar_frames=5)
m_det = evaluate_channels(gt, det_preds, num_frames=res.num_frames, collar_frames=5)

print('=== DOUBLE TALK BENCHMARK (Collar 50 ms) ===')
print(f"Baseline Macro F1: {m_base['macro_f1']:.4f}  |  Detector Macro F1: {m_det['macro_f1']:.4f}")
print(f"  Own   F1: Base = {m_base['by_label']['own']['f1']:.3f} -> Detector = {m_det['by_label']['own']['f1']:.3f} (+{m_det['by_label']['own']['f1'] - m_base['by_label']['own']['f1']:.3f}!)")
print(f"  Leak  F1: Base = {m_base['by_label']['leak']['f1']:.3f} -> Detector = {m_det['by_label']['leak']['f1']:.3f}")
print(f"  Third F1: Base = {m_base['by_label']['third']['f1']:.3f} -> Detector = {m_det['by_label']['third']['f1']:.3f}")


## 4. Очистка аудио от наводки/эха (`clean.py`) и прослушивание сэмплов

Применяем `TelephoneAudioCleaner` и оцениваем косинусную близость эмбеддингов до и после очистки.

In [ ]:
cleaner = TelephoneAudioCleaner(detector=detector)
cleaned_audio = cleaner.clean_stereo(audio, sample_rate=sr)

emb_ref = compute_spectral_embedding(res.clean_references['A'])
emb_noisy = compute_spectral_embedding(audio[0])
emb_clean = compute_spectral_embedding(cleaned_audio[0])

sim_before = cosine_similarity(emb_noisy, emb_ref)
sim_after = cosine_similarity(emb_clean, emb_ref)

print('Voice Embedding Cosine Similarity to Pure Owner Reference:')
print(f'  До очистки (noisy telephone): {sim_before:.4f}')
print(f'  После очистки (cleaned):      {sim_after:.4f}')

print('\n--- Прослушивание Track A ---')
print('1. Чистый исходный голос владельца (Original clean s_A):')
display(Audio(res.clean_references['A'], rate=sr))

print('2. Зашумленная телефонная дорожка A (с эхом собеседника и кодеком):')
display(Audio(audio[0], rate=sr))

print('3. Очищенная дорожка A (после cleaner.py):')
display(Audio(cleaned_audio[0], rate=sr))


## 5. Профилирование скорости на 1 ядре CPU (Лимит RTF <= 1/15)

По ТЗ обработка 1 минуты аудио должна занимать <= 4 секунд (`torch.set_num_threads(1)`).

In [ ]:
_, rtf, passed = measure_rtf(
    detector.predict_intervals,
    audio,
    sample_rate=sr,
    audio_duration_sec=res.duration_sec,
    num_threads=1,
)

print(f'Измеренный RTF: {rtf:.4f} (Лимит: <= 0.0667 = 1/15)')
print(f'Время обработки 60 сек аудио: {rtf * 60:.2f} сек (Лимит: <= 4.0 сек)')
print(f"Статус прохождения лимита: {'PASSED (ЗАПАС В 3 РАЗА!)' if passed else 'FAILED'}")
